In [123]:
import pandas as pd

# load data
df = pd.read_csv(
    "sample_data/wolverhampton_certs.csv",
    dtype={"uprn": "string"},
    low_memory=False
)

# type conversion
df["inspection_date"] = pd.to_datetime(df["inspection_date"], errors="coerce")
df["lodgement_date"] = pd.to_datetime(df["lodgement_date"], errors="coerce")
df["lodgement_datetime"] = pd.to_datetime(df["lodgement_datetime"], errors="coerce")
df["lodgement_date_derived"] = pd.to_datetime(df["lodgement_datetime"].dt.normalize(), errors="coerce")

# Exploration of DQ for EPC Certificates

In [124]:
cert_counts = df["uprn"].value_counts()

print("Unique Properties:", df["uprn"].nunique())
print("Missing UPRN:", df["uprn"].isna().sum())
print("UPRNs with multiple certs:", (cert_counts > 1).sum())
print("Max certs for single UPRN:", cert_counts.max())

Unique Properties: 80046
Missing UPRN: 1162
UPRNs with multiple certs: 28578
Max certs for single UPRN: 142


In [125]:
print("Missing inspection_date:", df["inspection_date"].isna().sum())
print("Missing lodgement_date:", df["lodgement_date"].isna().sum())
print("Missing lodgement_datetime:", df["lodgement_datetime"].isna().sum())
print("lodgement date vs datetime:", df["lodgement_date"].ne(
    df["lodgement_datetime"].dt.normalize()
).sum())
print("inspection date later than lodgment date:",
      df["inspection_date"].gt(df["lodgement_date"]).sum())

print("inspection date later than lodgment date derived:",
      df["inspection_date"].gt(df["lodgement_date_derived"]).sum())

Missing inspection_date: 0
Missing lodgement_date: 0
Missing lodgement_datetime: 0
lodgement date vs datetime: 1755
inspection date later than lodgment date: 1544
inspection date later than lodgment date derived: 50


In [126]:
inspection_later_than_lodgement = df.loc[df["inspection_date"].gt(df["lodgement_date"]), ['address','lodgement_date', 'lodgement_datetime', 'inspection_date']]

inspection_later_than_lodgement.loc[inspection_later_than_lodgement["address"]
                                    == "55, Frost Street"]

,address,lodgement_date,lodgement_datetime,inspection_date
0,"55, Frost Street",2012-11-28,2013-02-26 08:30:47,2013-02-19


In [127]:
mask = df["inspection_date"] > df["lodgement_datetime"]
anomalies = df.loc[mask].copy()

In [130]:
anomalies["days_difference"] = (
    anomalies["inspection_date"] - anomalies["lodgement_datetime"]
).dt.total_seconds() / 86400

# anomalies.head()
anomalies["days_difference"].describe()

count    50.000000
mean      1.380208
std       0.868731
min       0.360208
25%       0.360208
50%       1.360208
75%       2.360208
max       2.360208
Name: days_difference, dtype: float64

In [131]:
anomalies.nlargest(
    5,
    "days_difference"
)[[
    "certificate_number",
    "inspection_date",
    "lodgement_datetime",
    "days_difference"
]]

,certificate_number,inspection_date,lodgement_datetime,days_difference
3383,0350-3617-4010-2020-7065,2020-09-23,2020-09-20 15:21:18,2.360208
20890,0350-2897-9010-2020-5041,2020-09-23,2020-09-20 15:21:18,2.360208
28266,2671-3000-3301-5510-0204,2020-09-23,2020-09-20 15:21:18,2.360208
29213,7030-2031-5000-0117-0226,2020-09-23,2020-09-20 15:21:18,2.360208
29357,0350-3117-4010-2020-4031,2020-09-23,2020-09-20 15:21:18,2.360208


In [132]:
anomalies["lodgement_datetime"].value_counts()

lodgement_datetime
2020-09-20 15:21:18    50
Name: count, dtype: int64

In [133]:
anomalies["inspection_date"].value_counts()


inspection_date
2020-09-23    19
2020-09-21    18
2020-09-22    13
Name: count, dtype: int64

In [ ]:
latest_certs = (
    df.dropna(
        subset=["uprn"],
    )
    .sort_values(
        by=["uprn", "inspection_date", "lodgement_datetime", "certificate_number"],
        ascending=[True, False, False, False],
    )
    .drop_duplicates(
        subset=["uprn"],
        keep="first",
    )
)

assert latest_certs["uprn"].is_unique
assert latest_certs["uprn"].notna().all()

In [184]:
max_inspection_dates = (
    df.groupby("uprn")["inspection_date"].max()
)

max_inspection_dates.head()

uprn
100031823681   2022-06-15
100031823682   2015-06-18
100031823686   2014-02-03
100031823688   2026-05-06
100031823691   2023-05-09
Name: inspection_date, dtype: datetime64[us]

In [185]:
latest_certs["uprn"].map(max_inspection_dates)

56137    2022-06-15
97723    2015-06-18
92466    2014-02-03
113550   2026-05-06
72043    2023-05-09
            ...    
29459    2019-08-08
96501    2010-03-08
76416    2021-07-09
35075    2021-07-09
7449     2021-07-09
Name: uprn, Length: 80046, dtype: datetime64[us]

In [188]:
selected_dates = latest_certs["inspection_date"]

expected_dates = latest_certs["uprn"].map(max_inspection_dates)

comparison = selected_dates == expected_dates

assert comparison.all()